In [49]:
import requests
import pytz
import pandas as pd
import redis
import yaml
from os.path import abspath
from market import TradisAdapter
from datetime import datetime, timezone, timedelta
from ib_insync import *


pd.options.display.width = 300
pd.options.display.max_rows = 1500
pd.options.display.max_columns = None
pd.options.display.max_colwidth = None
pd.options.display.expand_frame_repr = False


chicago_tz = pytz.timezone('America/Chicago')


# Функции загрузки данных должны 
# возвращать Dataframe с одинаковым 
# количеством колонок в одинаковом порядке


def get_paraquet():
    df = pd.read_parquet("../jup/URA.parquet")

    df = df.set_index("datetime")
    df.columns = ["o", "h", "l", "c", "v"]

    df = df.astype("float64").round(2)

    return df


def get_polygon(symbol):
    symbol = symbol.split(".")[0]
    f_name = f"../data/polygon_nyse/2022/{symbol}.csv"
    df = pd.read_csv(open(f_name))
    df["dt"] = pd.to_datetime(df["t"], unit="s")

    # df["dt"] = df["dt"].dt.tz_localize("UTC").dt.tz_convert("US/Eastern").dt.tz_localize(None)

    df = df.set_index("dt")
    df = df[["o", "h", "l", "c", "v"]]

    df = df.astype("float64").round(4)

    return df


def get_tradis(symbol, dt1, dt2):

    config = yaml.full_load(open(abspath("../config/bot.yaml")))

    redis_config = config["sources"]["redis_cloud"]
    redis_client = redis.Redis(**redis_config)

    class FakeSchedule:
        def is_rth(self, *args, **kwargs):
            return True

    tradis = TradisAdapter(redis_client)
    tradis.schedule = FakeSchedule()

    res = [ln[2] for ln in tradis.load([symbol], dt1, dt2)]
    df = pd.DataFrame(res)
    df = df.set_index("dt")
    df = df[["o", "h", "l", "c", "vol"]]
    df = df.dropna()
    df["vol"] = df["vol"] * 100

    df = df[df["vol"] > 0]

    return df[dt1:dt2]


def get_ib_tws(symbol, dt1, dt2):
    util.startLoop()
    pd.options.display.width = 100
    tz = pytz.timezone('America/New_York')
    clientId = 2
    port = 4001

    ticker, exchange = symbol.split(".")

    with IB() as ib:
        
        ib.connect(port=port, clientId=clientId)

        contract = Stock(ticker, primaryExchange=exchange, exchange="SMART")
        ib.qualifyContracts(contract)
        
        print(contract)

        res = ib.reqHistoricalData(
            contract, 
            endDateTime="", 
            durationStr="5 D", 
            barSizeSetting="1 min", 
            whatToShow="TRADES", 
            useRTH=False
        )

        df = util.df(res)

        df["date"] = df["date"].dt.tz_localize("US/Pacific").dt.tz_convert("UTC").dt.tz_localize(None)
        df = df.set_index("date")

        df = df[dt1:dt2]

        df = df[["open", "high", "low", "close", "volume"]]

        df = df[df["volume"] > 0]

        return df


def compare_columns(row):
    """
    Попарно сравнивает колонки.
    """
    lv = list(row.values)
    n = len(lv) // 2
    style = "background: #dac"
    styles = ["" if abs(lv[i] - lv[i+n]) <= 0 else style for i in range(n)]
    return styles * 2


dt1 = datetime(2022, 9, 7)
dt2 = dt1 + timedelta(hours=15)

print(dt1, dt2)

symbol = "ARKK.ARCA"

# df1 = get_paraquet()[dt1:dt2]
df2 = get_polygon(symbol)[dt1:dt2]  #.round(2)
df3 = get_tradis(symbol, dt1, dt2)
# df4 = get_ib_tws(symbol, dt1, dt2)

# df3

df = pd.concat([df2, df3], axis=1)  # df1.eq(df2)

df.columns = list(range(len(df.columns)))
res = df.style.apply(compare_columns, axis=1).format(precision=2)
res





2022-09-07 00:00:00 2022-09-07 15:00:00


,0,1,2,3,4,5,6,7,8,9
dt,,,,,,,,,,
2022-09-07 08:00:00,39.88,39.97,39.88,39.97,2674.00,39.88,39.97,39.88,39.97,1700.00
2022-09-07 08:01:00,40.01,40.01,40.00,40.00,2570.00,40.01,40.01,40.00,40.00,1900.00
2022-09-07 08:02:00,40.01,40.01,40.01,40.01,1671.00,40.01,40.01,40.01,40.01,900.00
2022-09-07 08:03:00,40.03,40.15,40.03,40.15,4719.00,40.03,40.15,40.03,40.15,2700.00
2022-09-07 08:05:00,40.20,40.20,40.10,40.10,3200.00,40.20,40.20,40.10,40.10,1800.00
2022-09-07 08:07:00,39.99,39.99,39.99,39.99,391.00,39.99,39.99,39.99,39.99,100.00
2022-09-07 08:10:00,39.94,39.94,39.94,39.94,219.00,39.94,39.94,39.94,39.94,100.00
2022-09-07 08:11:00,39.91,39.91,39.90,39.90,1311.00,39.91,39.91,39.90,39.90,1200.00
2022-09-07 08:13:00,39.98,39.98,39.98,39.98,620.00,39.98,39.98,39.98,39.98,300.00
